In [ ]:
!pip install -q google-generativeai pypdf pdfplumber sentence-transformers faiss-cpu tiktoken

In [ ]:
import os
import re
import json
import pickle
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
from functools import lru_cache
import warnings
warnings.filterwarnings('ignore')

# PDF & text processing
import pypdf
import pdfplumber

# AI & embeddings
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import tiktoken

# For file upload in Colab
from google.colab import files
from IPython.display import display, Markdown, HTML

print("✅ All packages installed and imported.")

✅ All packages installed and imported.


In [ ]:

from google.colab import userdata
import google.generativeai as genai
import tiktoken

# Retrieve API key from Colab secrets
try:
    GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ API key loaded from Colab secrets.")
except Exception as e:
    print("❌ Could not retrieve 'GOOGLE_API_KEY' from Colab secrets.")
    raise e

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# List all available models to find a suitable one
print("\n📋 Available models (filtered for generateContent):")
available_models = []
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        available_models.append(m.name)
        print(f"  - {m.name}")

# Choose the best model (prefer flash, then pro, then any)
preferred_models = [
    'models/gemini-1.5-flash',
    'models/gemini-1.5-pro',
    'models/gemini-pro',
    'models/gemini-1.0-pro'
]
model_name = None
for pref in preferred_models:
    if pref in available_models:
        model_name = pref
        break

if model_name is None and available_models:
    model_name = available_models[0]  # fallback to first available

if model_name:
    print(f"\n✅ Selected model: {model_name}")
    model = genai.GenerativeModel(model_name)
    # Quick test
    test_response = model.generate_content("Say 'API works'")
    print("✅ Gemini API connected:", test_response.text)
else:
    raise RuntimeError("No suitable Gemini model found. Check your API key permissions.")

# Tokenizer for length management
tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """Return token count of a text string."""
    return len(tokenizer.encode(text))

✅ API key loaded from Colab secrets.

📋 Available models (filtered for generateContent):
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-3-1b-it
  - models/gemma-3-4b-it
  - models/gemma-3-12b-it
  - models/gemma-3-27b-it
  - models/gemma-3n-e4b-it
  - models/gemma-3n-e2b-it
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-pro-preview
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3-pro-image-preview
  - models/nano-banana-pro-preview
  - models/gem

Build Hierarchical Tree Index from PDF (Vectorless Core)

In [ ]:
# TreeIndex class - Hierarchical document representation
# No chunking, no vectors – just structure preserving

@dataclass
class Node:
    """Tree node representing a section of the document."""
    title: str
    content: str          # raw text of this section (without children)
    children: List['Node'] = field(default_factory=list)
    level: int = 0
    page: int = 0

    def to_dict(self):
        return {
            'title': self.title,
            'content': self.content[:500],
            'level': self.level,
            'page': self.page,
            'children': [c.to_dict() for c in self.children]
        }

class TreeIndex:
    """
    Builds a hierarchical tree from a PDF by detecting headings.
    Uses heuristics (font size, numbering, or text patterns).
    For simplicity, we use regex patterns for headings (e.g., "1.", "1.1", "Introduction").
    """
    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.root = Node(title="Document Root", content="", level=-1)
        self.nodes_by_path = {}   # for fast lookup

    def parse_pdf(self):
        """Extract text lines with page numbers using pdfplumber."""
        full_text = []
        pages_text = []
        with pdfplumber.open(self.pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                text = page.extract_text()
                if text:
                    lines = text.split('\n')
                    for line in lines:
                        full_text.append((line.strip(), page_num))
                    pages_text.append((text, page_num))
        return full_text, pages_text

    def is_heading(self, line: str) -> Tuple[bool, int]:
        """
        Determine if a line is a heading and its level.
        Returns (is_heading, level). Level 1 for top headings, 2 for subheadings, etc.
        """
        line = line.strip()
        if not line:
            return False, 0
        # Pattern 1: Numbered headings like "1.", "1.1", "2.3.4"
        match = re.match(r'^(\d+(?:\.\d+)*)\.\s+', line)
        if match:
            num_parts = match.group(1).split('.')
            level = len(num_parts)
            return True, level
        # Pattern 2: Roman numerals or words like "Chapter 1", "Introduction"
        # For simplicity, also treat all-caps short lines as level 1 headings
        if len(line) < 100 and line.isupper() and line.count(' ') < 8:
            return True, 1
        return False, 0

    def build_tree(self):
        """Build tree by scanning lines and maintaining a stack of nodes."""
        lines, _ = self.parse_pdf()
        stack = []  # stack of nodes at each level
        current_node = self.root

        for text, page_num in lines:
            is_head, level = self.is_heading(text)
            if is_head:
                # Create a new section node
                new_node = Node(title=text, content="", level=level, page=page_num)
                # Adjust stack: pop until we reach parent level
                while stack and stack[-1].level >= level:
                    stack.pop()
                parent = stack[-1] if stack else self.root
                parent.children.append(new_node)
                stack.append(new_node)
                current_node = new_node
            else:
                # Add text content to the current section node
                if current_node.content:
                    current_node.content += " " + text
                else:
                    current_node.content = text
        self._post_process()
        return self.root

    def _post_process(self):
        """Clean content: remove excessive whitespace, trim."""
        def clean(node):
            node.content = re.sub(r'\s+', ' ', node.content).strip()
            for child in node.children:
                clean(child)
        clean(self.root)

    def print_tree(self, node=None, depth=0):
        """Visualize the tree (for debugging)."""
        if node is None:
            node = self.root
        indent = "  " * depth
        print(f"{indent}{node.title} (page {node.page}) - {len(node.content)} chars")
        for child in node.children:
            self.print_tree(child, depth+1)

    def get_all_sections(self, node=None) -> List[Dict]:
        """Flatten tree into list of sections with path."""
        if node is None:
            node = self.root
        sections = []
        path = node.title
        if node.content and node.level >= 0:
            sections.append({
                'title': node.title,
                'path': path,
                'content': node.content,
                'level': node.level,
                'page': node.page
            })
        for child in node.children:
            sections.extend(self.get_all_sections(child))
        return sections

Reasoning-based Retrieval (Vectorless)

In [ ]:
# Reasoning navigator - human-like retrieval over tree structure
# Uses LLM to decide which branch to follow, showing reasoning steps

class ReasoningRetriever:
    def __init__(self, tree: TreeIndex, model):
        self.tree = tree
        self.model = model
        self.reasoning_trace = []

    def retrieve(self, query: str, max_depth: int = 3) -> str:
        """
        Retrieve relevant content by recursively reasoning over the tree.
        Returns the content of the most relevant section.
        """
        self.reasoning_trace = []
        best_node = self._navigate(query, self.tree.root, depth=0, max_depth=max_depth)
        return best_node.content if best_node else ""

    def _navigate(self, query: str, node: Node, depth: int, max_depth: int) -> Optional[Node]:
        """Recursively decide which child to explore based on reasoning."""
        if depth > max_depth or not node.children:
            return node

        # Prepare description of children for LLM
        children_desc = []
        for i, child in enumerate(node.children):
            summary = child.content[:300] if child.content else "[Empty section]"
            children_desc.append(
                f"{i+1}. **{child.title}** (page {child.page}): {summary}..."
            )
        children_text = "\n".join(children_desc)

        # Prompt LLM to choose the most relevant child
        prompt = f"""
You are an expert document navigator. You need to find the section most relevant to the user's query.

Query: {query}

Current section: "{node.title}"
Its sub-sections:
{children_text}

Reason step by step which sub-section (by number) is most promising to find the answer.
If none seems relevant, respond with "NONE".

Return your answer in format:
Reasoning: <your step-by-step reasoning>
Choice: <number or NONE>
"""
        response = self.model.generate_content(prompt)
        text = response.text

        # Parse reasoning and choice
        reasoning = ""
        choice = "NONE"
        for line in text.split('\n'):
            if line.lower().startswith("reasoning:"):
                reasoning = line[len("reasoning:"):].strip()
            elif line.lower().startswith("choice:"):
                choice = line[len("choice:"):].strip()

        self.reasoning_trace.append({
            'node': node.title,
            'depth': depth,
            'reasoning': reasoning,
            'choice': choice
        })

        if choice.upper() == "NONE":
            return node

        try:
            idx = int(choice) - 1
            if 0 <= idx < len(node.children):
                return self._navigate(query, node.children[idx], depth+1, max_depth)
        except:
            pass
        return node

    def explain_retrieval(self):
        """Print the reasoning path taken."""
        print("\n🔍 **Reasoning Trace:**")
        for step in self.reasoning_trace:
            print(f"📁 At '{step['node']}' (depth {step['depth']}):")
            print(f"   🤔 Reasoning: {step['reasoning']}")
            print(f"   ✅ Chose: {step['choice']}")

Answer Generation (Vectorless RAG)

In [ ]:
# Generate final answer using retrieved context + reasoning

class VectorlessRAG:
    def __init__(self, retriever: ReasoningRetriever, model):
        self.retriever = retriever
        self.model = model

    def answer(self, query: str) -> Dict[str, Any]:
        """Retrieve relevant section and generate answer with citations."""
        context = self.retriever.retrieve(query)
        trace = self.retriever.reasoning_trace

        if not context:
            return {
                "answer": "No relevant section found.",
                "context": "",
                "reasoning_path": trace
            }

        prompt = f"""
You are a helpful assistant. Use the following document section to answer the query.
If the section doesn't contain the answer, say "Information not available in the given document."

Query: {query}

Document Section:
\"\"\"
{context}
\"\"\"

Answer concisely and accurately, referencing the section title if possible.
"""
        response = self.model.generate_content(prompt)
        return {
            "answer": response.text,
            "context": context,
            "reasoning_path": trace
        }

Vector RAG (Baseline for Comparison)

In [ ]:
# Baseline – Traditional Vector RAG with chunking + FAISS

class VectorRAG:
    def __init__(self, embedding_model_name='all-MiniLM-L6-v2', chunk_size=500, overlap=50):
        self.embedder = SentenceTransformer(embedding_model_name)
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.index = None
        self.chunks = []   # list of (chunk_text, metadata)

    def build_from_tree(self, tree: TreeIndex):
        """
        Build vector index from the same tree by flattening sections and chunking.
        This is for fair comparison – uses the same document structure but then chunks.
        """
        sections = tree.get_all_sections()
        all_text = []
        for sec in sections:
            all_text.append(f"[{sec['title']}] {sec['content']}")
        full_doc = "\n\n".join(all_text)

        # Simple sliding window chunking
        words = full_doc.split()
        for i in range(0, len(words), self.chunk_size - self.overlap):
            chunk = " ".join(words[i:i+self.chunk_size])
            if chunk:
                self.chunks.append(chunk)
        print(f"Created {len(self.chunks)} chunks.")

        # Create embeddings and FAISS index
        embeddings = self.embedder.encode(self.chunks, show_progress_bar=True)
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings.astype(np.float32))

    def retrieve(self, query: str, k: int = 3) -> List[str]:
        """Retrieve top-k relevant chunks."""
        query_emb = self.embedder.encode([query])
        distances, indices = self.index.search(query_emb.astype(np.float32), k)
        return [self.chunks[idx] for idx in indices[0] if idx != -1]

    def answer(self, query: str, model, k: int = 3) -> Dict:
        chunks = self.retrieve(query, k)
        context = "\n---\n".join(chunks)
        prompt = f"""
You are a helpful assistant. Use the following document chunks to answer the query.
If they don't contain the answer, say "Information not available."

Query: {query}

Context:
{context}

Answer:
"""
        response = model.generate_content(prompt)
        return {
            "answer": response.text,
            "context": context,
            "retrieved_chunks": chunks
        }

Upload & Process PDF – Build Both Systems

In [ ]:
# Upload PDF and build the tree index

print("📁 Please upload your PDF file (e.g., research paper, manual, book chapter).")
uploaded = files.upload()

# Get filename
pdf_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {pdf_filename}")

# Build TreeIndex (Vectorless)
print("\n📚 Building hierarchical tree index from PDF...")
tree = TreeIndex(pdf_filename)
tree.build_tree()
print("Tree structure:")
tree.print_tree()  # shows first few levels

# Initialize Vectorless RAG
reasoning_retriever = ReasoningRetriever(tree, model)
vectorless_rag = VectorlessRAG(reasoning_retriever, model)

# Initialize Vector RAG baseline
print("\n⚡ Building Vector RAG baseline (chunking + FAISS)...")
vector_rag = VectorRAG(chunk_size=400, overlap=50)
vector_rag.build_from_tree(tree)

print("\n✅ Both systems ready.")

📁 Please upload your PDF file (e.g., research paper, manual, book chapter).


Saving 3_Lecture_Slides.pdf to 3_Lecture_Slides (2).pdf
✅ Uploaded: 3_Lecture_Slides (2).pdf

📚 Building hierarchical tree index from PDF...
Tree structure:
Document Root (page 0) - 0 chars
  EC 8203 (page 1) - 7986 chars
  1. Point to Point (page 12) - 0 chars
  2. Publish and Subscribe (page 12) - 5918 chars
  1. Kafka handles massive scale, processing millions of events per second, such as Netflix (page 25) - 40 chars
  2. It is durable and fault-tolerant, replicating data across brokers so no messages are lost, (page 25) - 39 chars
  3. Kafka offers flexible integrations, connecting easily with databases, data lakes, and (page 25) - 76 chars
  4. It supports real-time and batch data, enabling instant analytics for fraud detection and (page 25) - 33 chars
  5. Its publish-subscribe model allows multiple consumers to independently read the same (page 25) - 65 chars
  6. Kafka scales horizontally, letting organizations add brokers and partitions as (page 25) - 68 chars
  7. The Schema

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Created 7 chunks.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Both systems ready.


Query & Compare Both Approaches

In [ ]:
# Run a query and compare outputs

def run_comparison(query: str):
    print("="*80)
    print(f"🔍 QUERY: {query}\n")

    # Vectorless RAG
    print("🧠 VECTORLESS RAG (Reasoning-based retrieval):")
    result_vl = vectorless_rag.answer(query)
    print(f"Answer: {result_vl['answer']}")
    print("\nReasoning Path:")
    for step in result_vl['reasoning_path']:
        print(f"  - {step['node']} → chose {step['choice']} | {step['reasoning'][:100]}")
    print("\n" + "-"*50)

    # Vector RAG
    print("⚡ VECTOR RAG (Similarity-based retrieval):")
    result_v = vector_rag.answer(query, model, k=3)
    print(f"Answer: {result_v['answer']}")
    print(f"Retrieved {len(result_v['retrieved_chunks'])} chunks.")
    print("-"*80)

    # Optionally show context
    show_context = input("\nShow full contexts? (y/n): ").strip().lower()
    if show_context == 'y':
        print("\n--- Vectorless Context ---")
        print(result_vl['context'][:1000])
        print("\n--- Vector Context (first chunk) ---")
        print(result_v['context'][:1000])

# Example query – change to anything relevant to your PDF
query = "What is the main contribution of this document?"   # adjust as needed
run_comparison(query)

🔍 QUERY: What is the main contribution of this document?

🧠 VECTORLESS RAG (Reasoning-based retrieval):
Answer: The main contribution of this document is to define and explain core concepts and tools related to **Data Ingestion** and **Data Orchestration** within the context of Big Data Engineering. It covers what these processes entail, their key components, various types, and relevant architectural paradigms and platforms like the Decoupled Pipeline, Publish/Subscribe pattern, Distributed Log, Apache Airflow, and Apache Kafka.

Reasoning Path:
  - Document Root → chose 1 | 

--------------------------------------------------
⚡ VECTOR RAG (Similarity-based retrieval):
Answer: The main contribution of this document is a comprehensive explanation of **Kafka**, covering its core concepts, advanced features, use cases, ecosystem, and benefits. It also introduces foundational concepts like the Decoupled Pipeline Paradigm, Publish/Subscribe Pattern, and Distributed Log Concept, which underp

Interactive Query Loop

In [ ]:
# Interactive RAG Comparison with exit/continue option after each answer

print("\n🎯 Interactive RAG Comparison")
print("At any query prompt, type 'exit' or 'quit' to stop.")
print("After each answer, you'll be asked if you want to continue.\n")

while True:
    user_query = input("\n📝 Enter your query: ").strip()

    # Exit condition at query prompt
    if user_query.lower() in ['exit', 'quit']:
        print("👋 Exiting. Thanks for using Vectorless RAG!")
        break

    if not user_query:
        print("⚠️ Empty query. Please enter a valid question.")
        continue

    # Run the comparison (shows both answers)
    run_comparison(user_query)

    # Ask user whether to continue or exit
    while True:
        choice = input("\n🔁 Continue? (y/n): ").strip().lower()
        if choice in ['y', 'yes', 'n', 'no']:
            break
        print("Please answer 'y' (yes) or 'n' (no).")

    if choice in ['n', 'no']:
        print("👋 Exiting. Thanks for using Vectorless RAG!")
        break
    # else continue to next iteration


🎯 Interactive RAG Comparison
At any query prompt, type 'exit' or 'quit' to stop.
After each answer, you'll be asked if you want to continue.


📝 Enter your query: why kafka?
🔍 QUERY: why kafka?

🧠 VECTORLESS RAG (Reasoning-based retrieval):
Answer: Information not available in the given document.

Reasoning Path:
  - Document Root → chose 4 | The user is asking for the reasons or benefits of using Kafka. Sections 4 through 12 all provide dis

--------------------------------------------------
⚡ VECTOR RAG (Similarity-based retrieval):
Answer: Kafka is preferred because traditional message brokers often struggle with scale, fault tolerance, and replayability.

Kafka provides solutions to these issues by offering:
*   **High throughput & low latency:** Capable of processing millions of events per second.
*   **Durability:** Messages are stored on disk and replicated across brokers to prevent data loss.
*   **Scalability:** It can scale horizontally by adding brokers and partitions as wo

Evaluation & Export Results

In [ ]:
# Evaluate on a set of questions and export reasoning traces

# Example: define a list of test questions (tailor to your PDF)
test_questions = [
    "What problem does this paper solve?",
    "What method is proposed?",
    "What are the main results?"
]

results = []
for q in test_questions:
    # Vectorless
    vl_res = vectorless_rag.answer(q)
    # Vector
    v_res = vector_rag.answer(q, model)

    results.append({
        "query": q,
        "vectorless_answer": vl_res['answer'],
        "vectorless_reasoning": vl_res['reasoning_path'],
        "vector_answer": v_res['answer']
    })

# Save to JSON
with open("rag_comparison_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("✅ Results saved to rag_comparison_results.json")

# Display summary
for r in results:
    print("\n" + "="*60)
    print(f"Query: {r['query']}")
    print(f"Vectorless Answer: {r['vectorless_answer'][:200]}...")
    print(f"Vector Answer: {r['vector_answer'][:200]}...")